# MobileNet Inference with TaskVine Serverless Functions

This notebook classifies a manifest-defined image dataset with TaskVine's
Function Library and Function Call model. Each library process loads MobileNet
once, then reuses that ONNX Runtime session across image microbatches.

For isolated tasks that load a new model session per batch, compare this
notebook with `mobilenet-python-task.ipynb`.


## Step 1 — Configure the workload and library

The dataset is divided into deterministic microbatches. `LIBRARY_NAME`
identifies the persistent function library installed on TaskVine workers.


In [ ]:
BATCH_SIZE = 4
TOP_K = 5
LIBRARY_NAME = "mobilenetv2-inference"


## Step 2 — Import helpers and locate staged inputs

Floability runs this notebook inside the staged `workflow/` directory. The
paths below refer to data that Floability copied or downloaded according to
`data/data.yml`.

The workflow-specific helper module is registered by value so TaskVine can
serialize the library initializer and callable function for worker execution.


In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

import cloudpickle
from IPython.display import display
from PIL import Image

import mobilenet_helpers

cloudpickle.register_pickle_by_value(mobilenet_helpers)

if os.environ.get("FLOABILITY_WORKERS_ENABLED") == "0":
    raise RuntimeError("This serverless notebook requires Floability workers")

MODEL_PATH = Path("data/mobilenetv2-10.onnx")
LABELS_PATH = Path("data/imagenet-synset.txt")
IMAGE_DIR = Path("data/images")
MANIFEST_PATH = Path("data/image-manifest.json")
OUTPUT_DIR = Path("outputs")


## Step 3 — Validate the dataset contract and create batches

The manifest defines the dataset identity, image count, filenames, and SHA-256
checksums. The workflow validates it before installing the library or
submitting calls.


In [ ]:
for required_path in (MODEL_PATH, LABELS_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required staged input not found: {required_path}")

image_paths, manifest = mobilenet_helpers.load_and_verify_images(
    IMAGE_DIR,
    MANIFEST_PATH,
)
image_batches = mobilenet_helpers.make_image_batches(image_paths, BATCH_SIZE)
expected_image_names = [path.name for path in image_paths]

print(f"Dataset: {manifest['dataset_id']}@{manifest['dataset_version']}")
print(f"Verified images: {len(image_paths)}")
print(f"Microbatches: {len(image_batches)}")


## Step 4 — Create the TaskVine manager and declare inputs

Floability supplies a unique manager name and the permitted manager-port range.
The notebook creates the manager, materializes one directory per image batch,
and declares the model, labels, and batches as TaskVine inputs.


In [ ]:
import ndcctools.taskvine as vine

manager_name = os.environ.get("VINE_MANAGER_NAME")
if not manager_name:
    raise RuntimeError("VINE_MANAGER_NAME is not set; run through Floability")

port_spec = os.environ.get("VINE_MANAGER_PORTS", "9123,9150")
ports = [int(value.strip()) for value in port_spec.split(",") if value.strip()]
if not ports:
    raise ValueError("VINE_MANAGER_PORTS does not contain a port")
manager_port = ports[0] if len(ports) == 1 else [min(ports), max(ports)]

manager = vine.Manager(port=manager_port, name=manager_name)
manager.tune("watch-library-logfiles", 1)

batch_root, batch_paths = mobilenet_helpers.materialize_batch_directories(
    image_batches,
    prefix="mobilenet-serverless-batches-",
)
declared_model = manager.declare_file(str(MODEL_PATH), cache=True)
declared_labels = manager.declare_file(str(LABELS_PATH), cache=True)
declared_batches = {
    batch_path: manager.declare_file(str(batch_path), cache=True)
    for batch_path in batch_paths
}

print(f"Manager name: {manager_name}")
print(f"Manager port: {manager.port}")
print("Declared the model, labels, and image microbatches")


## Step 5 — Install the persistent function library

`initialize_mobilenet_library` runs once for each library process and returns
the reusable ONNX session and labels. `classify_batch_with_shared_session` is
the callable function. It retrieves that state instead of loading the model
again for every batch.


In [ ]:
library = manager.create_library_from_functions(
    LIBRARY_NAME,
    mobilenet_helpers.classify_batch_with_shared_session,
    add_env=False,
    exec_mode="direct",
    library_context_info=[
        mobilenet_helpers.initialize_mobilenet_library,
        ["model.onnx", "labels.txt"],
        {},
    ],
)
library.add_input(declared_model, "model.onnx")
library.add_input(declared_labels, "labels.txt")
library.set_cores(1)
library.set_function_slots(1)
manager.install_library(library)

print(f"Installed persistent library: {LIBRARY_NAME}")


## Step 6 — Submit one Function Call per batch

A Function Call names a function installed in the library. Each call receives
only its image batch and lightweight arguments; the model session and labels
remain inside the persistent library process.


In [ ]:
task_batches = {}
started_at = time.perf_counter()

for batch_path in batch_paths:
    task = vine.FunctionCall(
        LIBRARY_NAME,
        "classify_batch_with_shared_session",
        "batch",
        TOP_K,
    )
    task.add_input(declared_batches[batch_path], "batch")
    task.set_cores(1)

    task_id = manager.submit(task)
    task_batches[task_id] = batch_path.name

print(f"Submitted {len(task_batches)} Function Calls")


## Step 7 — Collect calls and verify library execution

Each result records its library load ID and process ID. Matching library and
function process IDs confirms that the call executed inside the persistent
library process. Repeated load IDs show state reuse across calls.


In [ ]:
results = []
failures = []

while not manager.empty():
    completed = manager.wait(5)
    if not completed:
        continue
    if not completed.successful():
        failures.append((completed.id, completed.result))
        print(f"FAILED task={completed.id} result={completed.result}")
        continue
    if isinstance(completed.output, Exception):
        failures.append((completed.id, repr(completed.output)))
        print(f"FAILED task={completed.id} exception={completed.output!r}")
        continue

    result = completed.output
    if result["function_pid"] != result["library_pid"]:
        raise RuntimeError("Function Call did not run inside its library process")

    result["task_id"] = completed.id
    result["batch"] = task_batches[completed.id]
    result["worker_address"] = completed.addrport
    results.append(result)
    print(
        f"task={completed.id} batch={result['batch']} "
        f"load_id={result['library_load_id']} worker={completed.addrport}"
    )

if failures:
    raise RuntimeError(f"Inference call failures: {failures}")
if len(results) != len(batch_paths):
    raise RuntimeError(
        f"Expected {len(batch_paths)} call results; received {len(results)}"
    )

elapsed_seconds = time.perf_counter() - started_at
library_reuse = mobilenet_helpers.group_batches_by_library_load(results)

print(f"Execution completed in {elapsed_seconds:.2f} seconds")
print(f"Persistent library instances: {len(library_reuse)}")
for load_id, batches in sorted(library_reuse.items()):
    print(f"  {load_id}: {len(batches)} microbatch(es)")


## Step 8 — Validate and save the output

Every manifest image must appear exactly once. The summary preserves each
call's library identity so state reuse remains visible after the run.


In [ ]:
predictions = mobilenet_helpers.predictions_by_image(
    results,
    expected_image_names,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "dataset_id": manifest["dataset_id"],
    "dataset_version": manifest["dataset_version"],
    "execution_mode": "stateful-serverless",
    "execution_mode_source": "mobilenet-serverless-taskvine.ipynb",
    "image_count": len(image_paths),
    "batch_size": BATCH_SIZE,
    "batch_count": len(image_batches),
    "elapsed_seconds": elapsed_seconds,
    "distinct_library_loads": len(library_reuse),
    "task_results": results,
}
summary_path = OUTPUT_DIR / "stateful-serverless-summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

contact_sheet_path = OUTPUT_DIR / "stateful-serverless-contact-sheet.jpg"
displayed_image_count = mobilenet_helpers.save_contact_sheet(
    image_paths,
    predictions,
    contact_sheet_path,
)

shutil.rmtree(batch_root)

print("=" * 72)
print("MOBILENET SERVERLESS INFERENCE COMPLETE")
print(f"Validated images: {len(predictions)}")
print(f"Persistent library instances: {len(library_reuse)}")
print(f"Elapsed time: {elapsed_seconds:.2f} seconds")
print(f"Results: {summary_path}")
print(f"Contact sheet: {contact_sheet_path} ({displayed_image_count} images shown)")
print("=" * 72)

display(Image.open(contact_sheet_path))


## Interpretation

The model is initialized once per TaskVine library process rather than once per
microbatch. Multiple load IDs are normal when multiple workers host library
instances. Repeated IDs across batches demonstrate reuse within an instance.
This small run demonstrates behavior; it is not a performance benchmark.
